# Working Capital Benchmarking: Indian Capital Goods

**Question:** how much cash is each company tying up in working capital compared with its peers?

**Method in one line:** standard DSO / DIO / DPO / CCC from Yahoo Finance statements, benchmarked against a leave-one-out peer median, with the rupee 'prize' shown as a range. No calibration or scaling factors are applied anywhere. Limitations are listed at the end.

In [ ]:
!pip -q install yfinance openpyxl plotly
import os, time, pandas as pd, numpy as np, yfinance as yf
from google.colab import drive

try:
    drive.mount('/content/drive')          # no force_remount
    BASE = '/content/drive/MyDrive/working_capital'
except Exception as e:
    print(f"Drive not mounted ({e}). Using local storage; files are LOST when the runtime resets.")
    BASE = '/content/working_capital'

os.makedirs(BASE, exist_ok=True)
print('Using working directory:', BASE)

Mounted at /content/drive
Using working directory: /content/drive/MyDrive/working_capital


## 1. Companies and peer groups
Benchmarking is done **within** a peer group, because order-book businesses (defence, EPC) have structurally long inventory and receivable cycles compared with product manufacturers. Verify each Yahoo ticker returns data in the next step, because some symbols may be wrong.

In [ ]:
COMPANIES = {   # name: (ticker, peer_group)
 'L&T':            ('LT.NS',         'Order-book driven'),
 'HAL':            ('HAL.NS',        'Order-book driven'),
 'BEL':            ('BEL.NS',        'Order-book driven'),
 'BHEL':           ('BHEL.NS',       'Order-book driven'),
 'Siemens':        ('SIEMENS.NS',    'Electrical & industrial OEM'),
 'ABB India':      ('ABB.NS',        'Electrical & industrial OEM'),
 'Hitachi Energy': ('POWERINDIA.NS', 'Electrical & industrial OEM'),
 'GE Vernova T&D': ('GVT&D.NS',      'Electrical & industrial OEM'),
 'CG Power':       ('CGPOWER.NS',    'Electrical & industrial OEM'),
 'Cummins':        ('CUMMINSIND.NS', 'Electrical & industrial OEM'),
 'Polycab':        ('POLYCAB.NS',    'Electrical & industrial OEM'),
}

## 2. Download and cache (single fetch, raw file saved as the audit trail)

In [ ]:
def pick(frame, names):
    for n in names:
        if n in frame.index:
            return frame.loc[n]
    return pd.Series(dtype='float64')

recs = []
for name, (tkr, grp) in COMPANIES.items():
    t = yf.Ticker(tkr)
    fin, bs = t.financials, t.balance_sheet
    if fin.empty or bs.empty:
        print('NO DATA ->', name, tkr)
        continue
    d = pd.concat({
        'revenue':     pick(fin, ['Total Revenue', 'Operating Revenue']),
        'cogs':        pick(fin, ['Cost Of Revenue', 'Reconciled Cost Of Revenue']),
        'receivables': pick(bs,  ['Accounts Receivable', 'Receivables']),
        'inventory':   pick(bs,  ['Inventory']),
        'payables':    pick(bs,  ['Accounts Payable', 'Payables']),
    }, axis=1)
    d['company'], d['group'] = name, grp
    d['fy_end'] = pd.to_datetime(d.index)
    recs.append(d.reset_index(drop=True))
    time.sleep(1.5)

df = pd.concat(recs, ignore_index=True)
df['fy'] = df['fy_end'].dt.year
df.to_csv(f'{BASE}/raw_financials.csv', index=False)
print('Saved', len(df), 'rows to', f'{BASE}/raw_financials.csv')

Saved 54 rows to /content/drive/MyDrive/working_capital/raw_financials.csv


## 3. Data-quality check
How many periods does each company have, and which inputs are missing? Rows with any missing input are dropped from the analysis. Report this in the README.

In [ ]:
dq = df.groupby('company').agg(periods=('fy', 'count'),
                               first_period_end=('fy_end', 'min'),
                               latest_period_end=('fy_end', 'max'))
dq['periods_with_all_inputs'] = (df.dropna(subset=['revenue','cogs','receivables','inventory','payables'])
                                   .groupby('company')['fy'].count())
dq['missing_cells'] = df.isna().groupby(df['company']).sum()[['revenue','cogs','receivables','inventory','payables']].sum(axis=1)
dq

,periods,first_period_end,latest_period_end,periods_with_all_inputs,missing_cells
company,,,,,
ABB India,5,2021-12-31,2025-12-31,4,5
BEL,5,2022-03-31,2026-03-31,4,5
BHEL,5,2022-03-31,2026-03-31,4,5
CG Power,5,2022-03-31,2026-03-31,4,5
Cummins,5,2022-03-31,2026-03-31,4,5
GE Vernova T&D,5,2022-03-31,2026-03-31,4,5
HAL,5,2022-03-31,2026-03-31,4,5
Hitachi Energy,5,2021-12-31,2026-03-31,4,5
L&T,5,2022-03-31,2026-03-31,4,5


## 4. Metrics: standard definitions, year-end balances, no scaling factors
- DSO = receivables / revenue x 365
- DIO = inventory / COGS x 365
- DPO = payables / COGS x 365
- CCC = DSO + DIO - DPO

Year-end balances are used for every year so that all periods are treated the same way.

In [ ]:
ok = (df.dropna(subset=['revenue','cogs','receivables','inventory','payables'])
        .sort_values(['company','fy_end']).reset_index(drop=True))

ok['DSO'] = ok.receivables / ok.revenue * 365
ok['DIO'] = ok.inventory   / ok.cogs    * 365
ok['DPO'] = ok.payables    / ok.cogs    * 365
ok['CCC'] = ok.DSO + ok.DIO - ok.DPO
ok[['company','fy_end','DSO','DIO','DPO','CCC']].round(1).tail(15)

,company,fy_end,DSO,DIO,DPO,CCC
29,Hitachi Energy,2024-03-31,109.0,87.6,178.5,18.1
30,Hitachi Energy,2025-03-31,92.6,78.2,172.4,-1.6
31,Hitachi Energy,2026-03-31,88.2,103.8,215.4,-23.4
32,L&T,2023-03-31,89.9,21.4,155.8,-44.6
33,L&T,2024-03-31,81.2,16.5,133.0,-35.3
34,L&T,2025-03-31,77.1,16.4,111.9,-18.4
35,L&T,2026-03-31,77.6,18.0,131.3,-35.7
36,Polycab,2023-03-31,32.6,99.6,27.2,105.0
37,Polycab,2024-03-31,42.1,97.6,26.6,113.1
38,Polycab,2025-03-31,43.0,77.1,30.1,90.0


## 5. Benchmark against a leave-one-out peer median, then size the prize
Each company is compared with the median of its **peers only** (itself excluded), using each company's **latest available period**. Fiscal year-ends differ across companies, so the period end is shown in the output. Do not read differences of a few days as meaningful.

The prize is in Rs crore:
- **Gap to median:** cash released if the company matched the peer median on each lever (upper bound, not a forecast).
- **Conservative case:** 25% of that gap. Use this as the headline.
- Levers where the company already beats the median contribute zero.

In [ ]:
latest = ok.sort_values('fy_end').groupby('company').tail(1).set_index('company').copy()

def loo_median(s):
    # median of the other companies in the group (the company itself excluded)
    return pd.Series([s.drop(i).median() for i in s.index], index=s.index)

for m in ['DSO', 'DIO', 'DPO']:
    latest[f'{m}_peer'] = latest.groupby('group')[m].transform(loo_median)
latest['peer_n'] = latest.groupby('group')['DSO'].transform('count') - 1

CR = 1e7   # Yahoo reports in rupees
latest['prize_DSO'] = (latest.DSO - latest.DSO_peer).clip(lower=0) * latest.revenue / 365 / CR
latest['prize_DIO'] = (latest.DIO - latest.DIO_peer).clip(lower=0) * latest.cogs    / 365 / CR
latest['prize_DPO'] = (latest.DPO_peer - latest.DPO).clip(lower=0) * latest.cogs    / 365 / CR
latest['prize_gap_to_median'] = latest[['prize_DSO','prize_DIO','prize_DPO']].sum(axis=1)
latest['prize_conservative']  = latest.prize_gap_to_median * 0.25

# flag results that rest on an extreme gap, which usually reflects business model rather than inefficiency
latest['outlier_flag'] = ((latest.DIO > 2 * latest.DIO_peer) | (latest.DSO > 2 * latest.DSO_peer)).map({True: 'check: >2x peer median', False: ''})

res = (latest.reset_index()
       .sort_values('prize_conservative', ascending=False)
       [['company','group','fy_end','peer_n','DSO','DIO','DPO','CCC',
         'prize_DSO','prize_DIO','prize_DPO','prize_gap_to_median','prize_conservative','outlier_flag']]
       .round(1))
res

,company,group,fy_end,peer_n,DSO,DIO,DPO,CCC,prize_DSO,prize_DIO,prize_DPO,prize_gap_to_median,prize_conservative,outlier_flag
2,HAL,Order-book driven,2026-03-31,3,46.7,564.5,81.1,530.1,0.0,19277.3,2745.0,22022.3,5505.6,check: >2x peer median
4,BEL,Order-book driven,2026-03-31,3,175.4,264.6,93.2,346.8,7247.9,2031.1,1467.0,10746.1,2686.5,check: >2x peer median
8,Polycab,Electrical & industrial OEM,2026-03-31,6,48.5,91.6,29.6,110.5,0.0,850.9,6124.5,6975.3,1743.8,
6,GE Vernova T&D,Electrical & industrial OEM,2026-03-31,6,127.9,131.5,169.2,90.1,719.4,503.5,0.0,1222.8,305.7,
9,Cummins,Electrical & industrial OEM,2026-03-31,6,84.1,54.1,76.5,61.8,0.0,0.0,1165.8,1165.8,291.4,
5,L&T,Order-book driven,2026-03-31,3,77.6,18.0,131.3,-35.7,767.8,0.0,0.0,767.8,191.9,
7,CG Power,Electrical & industrial OEM,2026-03-31,6,86.9,67.2,104.6,49.4,25.3,0.0,609.8,635.1,158.8,
0,Siemens,Electrical & industrial OEM,2025-09-30,6,62.7,60.4,112.9,10.1,0.0,0.0,443.5,443.5,110.9,
3,Hitachi Energy,Electrical & industrial OEM,2026-03-31,6,88.2,103.8,215.4,-23.4,58.0,375.8,0.0,433.8,108.5,
1,ABB India,Electrical & industrial OEM,2025-12-31,6,88.7,88.0,148.1,28.6,115.1,200.8,0.0,315.9,79.0,


## 6. Validation against an independent source (run the cell below, then write your note)
Screener (consolidated, Mar 2026) figures are entered for HAL, L&T and Cummins. Add Polycab and BEL from their Screener pages to complete the set.

**Expect real gaps, and explain them.** Providers do not define these ratios the same way. For example, Screener shows Cummins inventory days of 56, while another provider (Value Research) reports about 32 for the same year because it divides average inventory by sales rather than COGS. L&T is an EPC company: Screener shows 265 days payable against 77 debtor days, which gives a strongly negative cash conversion cycle (-152). Customer advances and project payables make that figure a feature of the business model, not a sign of superior efficiency. The goal is to measure the gap and say why it exists, not to remove it.


In [ ]:
screener = {   # company: (DSO, DIO, DPO, CCC) from Screener.in 'Ratios' section, CONSOLIDATED, Mar 2026
    'HAL':            (45, 859, 116, 788),            # as shown on Screener
    'L&T':            (77, 36, 265, 77 + 36 - 265),   # CCC computed from the three Screener rows
    'Cummins':        (83, 56, 79, 83 + 56 - 79),     # CCC computed from the three Screener rows
    'Polycab':        (None, None, None, None),       # TODO: copy from screener.in/company/POLYCAB/consolidated/
    'BEL':            (None, None, None, None),       # TODO: copy from screener.in/company/BEL/consolidated/
}
rows = []
for comp, vals in screener.items():
    if comp not in latest.index or vals[0] is None:
        continue
    for metric, ref in zip(['DSO','DIO','DPO','CCC'], vals):
        mine = latest.loc[comp, metric]
        rows.append({'company': comp, 'metric': metric, 'screener': ref,
                     'mine': round(mine, 1), 'abs_diff': round(abs(mine - ref), 1),
                     'pct_diff': round((mine - ref) / ref * 100, 1)})
check = pd.DataFrame(rows)
check


,company,metric,screener,mine,abs_diff,pct_diff
0,HAL,DSO,45,46.7,1.7,3.7
1,HAL,DIO,859,564.5,294.5,-34.3
2,HAL,DPO,116,81.1,34.9,-30.1
3,HAL,CCC,788,530.1,257.9,-32.7
4,L&T,DSO,77,77.6,0.6,0.8
5,L&T,DIO,36,18.0,18.0,-49.9
6,L&T,DPO,265,131.3,133.7,-50.4
7,L&T,CCC,-152,-35.7,116.3,-76.5
8,Cummins,DSO,83,84.1,1.1,1.4
9,Cummins,DIO,56,54.1,1.9,-3.3


**Write your validation note here after running the cell above** (what matched, what did not, and why, for example: "Yahoo's Cost of Revenue includes X, so DIO is higher than Screener's").

In [ ]:
import plotly.express as px
long = res.melt(id_vars=['company'], value_vars=['prize_DSO','prize_DIO','prize_DPO'],
                var_name='lever', value_name='Rs Cr')
px.bar(long, x='company', y='Rs Cr', color='lever',
       title='Cash release if the company matched its peer median (upper bound, Rs Cr)').show()

px.bar(res, x='company', y='prize_conservative',
       title='Conservative case: 25% of the gap closed (Rs Cr)').show()

with pd.ExcelWriter(f'{BASE}/working_capital_benchmark.xlsx') as xw:
    ok.to_excel(xw, sheet_name='all_years', index=False)
    res.to_excel(xw, sheet_name='latest_prize', index=False)
    check.to_excel(xw, sheet_name='validation', index=False)
    dq.to_excel(xw, sheet_name='data_quality')
print('Saved:', f'{BASE}/working_capital_benchmark.xlsx')

Saved: /content/drive/MyDrive/working_capital/working_capital_benchmark.xlsx


## 7. README

The next cell writes this text to `README.md` in your working folder.

In [ ]:
readme = '''# Working Capital Benchmarking: Indian Capital Goods

## Question
How much cash is each company tying up in working capital relative to its peers?

## Method
- Source: Yahoo Finance annual statements via `yfinance` (unofficial; raw download saved to `raw_financials.csv`).
- Metrics (standard definitions, year-end balances, **no scaling or calibration factors**):
  DSO = receivables / revenue x 365; DIO = inventory / COGS x 365; DPO = payables / COGS x 365; CCC = DSO + DIO - DPO.
- Benchmark: median of the other companies in the same peer group (leave-one-out), using each company's latest available period.
- Prize: rupees released if a company matched the peer median on each lever (upper bound), plus a conservative case of 25% of that gap.

## Validation
Results were compared with Screener.in for the companies listed in the validation sheet. Differences and their likely causes are described in the notebook.

## Limitations
- Small peer groups (3 to 7 comparators). Medians are unstable.
- Yahoo's 'Cost of Revenue' and Indian-GAAP definitions differ, so ratios will not match Screener exactly.
- Contract assets, customer advances and retention money distort DSO and DIO for EPC and defence companies.
- Fiscal year-ends differ across companies (some report on a non-March year-end), so the compared periods are not identical.
- yfinance returns only a few annual periods, so no long trend analysis is possible without annual-report data.
- The prize is an upper bound for discussion, not a forecast of achievable cash release.
'''
with open(f'{BASE}/README.md', 'w') as f:
    f.write(readme)
print('README written to', f'{BASE}/README.md')

README written to /content/drive/MyDrive/working_capital/README.md


In [ ]:
import json, datetime
ok.to_csv(f'{BASE}/all_years.csv', index=False)
res.to_csv(f'{BASE}/latest_prize.csv', index=False)
check.to_csv(f'{BASE}/validation.csv', index=False)
dq.reset_index().to_csv(f'{BASE}/data_quality.csv', index=False)
json.dump({'refreshed_utc': datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%d %H:%M UTC')},
          open(f'{BASE}/meta.json', 'w'))
print('Exported 5 files to', BASE)

Exported 5 files to /content/drive/MyDrive/working_capital
